# 🦷 Dental Disease Detection - YOLOv8 Training
## Dataset: 10,171 intraoral X-ray images
## Classes: Caries, Crown, Filling, Implant, Periapical-lesion
## Target: 80%+ mAP50

In [ ]:
# Install dependencies
!pip install ultralytics albumentations -q

import os
import shutil
from pathlib import Path
import yaml
import json
from datetime import datetime

# ENV Configuration
os.environ["PYTHONPATH"] = "/kaggle/working/dental-ai"
print("✅ Dependencies installed")

## 📁 Dataset Configuration
**IMPORTANT**: Dataset je dostupný ako `mojzosit` na Kaggle (tvoj vlastný dataset).

In [ ]:
# Dataset configuration - OPRAVENÉ
# Dataset je tvoj `mojzosit` - bude sa objaviť v /kaggle/input/mojzosit/mega-dataset/mega-dataset/
DATASET_PATH = "/kaggle/input/mojzosit/mega-dataset/mega-dataset"
WORK_DIR = "/kaggle/working/dental-ai"

# Create work directory
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/runs", exist_ok=True)

print(f"📁 Dataset: {DATASET_PATH}")
print(f"📁 Work dir: {WORK_DIR}")

# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    print("❌ Dataset not found!")
    print("Make sure you added 'mojzosit' as input to this notebook")
    dataset_dir = None
else:
    dataset_dir = DATASET_PATH
    print("✅ Dataset found")

In [ ]:
# Verify dataset structure
if dataset_dir:
    yaml_path = f"{dataset_dir}/data.yaml"
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            config = yaml.safe_load(f)
        print(f"\n📊 Dataset config:")
        print(f"  Classes: {config.get('nc', 'N/A')}")
        print(f"  Names: {config.get('names', 'N/A')}")
        
        # Count images
        for split in ['train', 'val', 'test']:
            img_dir = f"{dataset_dir}/{split}/images"
            if os.path.exists(img_dir):
                count = len(list(Path(img_dir).glob("*.jpg"))) + len(list(Path(img_dir).glob("*.png")))
                print(f"  {split}: {count} images")
    else:
        print(f"❌ data.yaml not found at {yaml_path}")
        print("Files in dataset:")
        for f in Path(dataset_dir).iterdir():
            print(f"  {f.name}")

## 🚀 Training Configuration
**OPRAVENÉ**: batch size zvýšený na 8 pre lepšiu GPU využitie na T4

In [ ]:
# Training configuration - OPRAVENÉ
CONFIG = {
    # Model
    "model": "yolov8x.pt",  # Use yolov8x for best accuracy
    
    # Training
    "epochs": 150,           # More epochs for better convergence
    "patience": 30,          # Early stopping
    "batch": 8,              # OPRAVENÉ: z 4 na 8 pre lepšiu GPU využitie
    "imgsz": 640,            # Start with 640, can increase later
    
    # Optimizer
    "optimizer": "AdamW",
    "lr0": 0.001,            # Initial learning rate
    "lrf": 0.01,             # Final learning rate (lr0 * lrf)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    
    # Augmentation
    "mosaic": 1.0,           # Mosaic augmentation
    "mixup": 0.15,           # Mixup augmentation
    "copy_paste": 0.1,       # Copy-paste augmentation
    "hsv_h": 0.015,          # HSV-Hue augmentation
    "hsv_s": 0.7,            # HSV-Saturation augmentation
    "hsv_v": 0.4,            # HSV-Value augmentation
    "flipud": 0.0,           # No vertical flip (dental X-rays)
    "fliplr": 0.5,           # Horizontal flip
    "erasing": 0.4,          # Random erasing
    
    # Other
    "device": 0,             # GPU
    "workers": 2,
    "project": f"{WORK_DIR}/runs",
    "name": f"dental_yolov8x_{datetime.now().strftime('%Y%m%d_%H%M')}",
}

print("⚙️ Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 🏋️ Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO
import torch

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU found! Training will be slow.")

# Initialize model
model = YOLO(CONFIG["model"])
print(f"\n🤖 Model: {CONFIG['model']}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M")

In [ ]:
# Start training
print("🏋️ Starting training...")
print(f"⏰ Estimated time: {CONFIG['epochs'] * 5 / 60:.1f} hours (based on ~5 min/epoch on T4)")
print("=" * 60)

results = model.train(
    data=f"{dataset_dir}/data.yaml",
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch"],
    imgsz=CONFIG["imgsz"],
    device=CONFIG["device"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    lrf=CONFIG["lrf"],
    momentum=CONFIG["momentum"],
    weight_decay=CONFIG["weight_decay"],
    patience=CONFIG["patience"],
    mosaic=CONFIG["mosaic"],
    mixup=CONFIG["mixup"],
    copy_paste=CONFIG["copy_paste"],
    hsv_h=CONFIG["hsv_h"],
    hsv_s=CONFIG["hsv_s"],
    hsv_v=CONFIG["hsv_v"],
    flipud=CONFIG["flipud"],
    fliplr=CONFIG["fliplr"],
    erasing=CONFIG["erasing"],
    workers=CONFIG["workers"],
    project=CONFIG["project"],
    name=CONFIG["name"],
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print("\n✅ Training complete!")

## 📊 Evaluate Results

In [ ]:
# Print results
print("=" * 60)
print("📊 TRAINING RESULTS")
print("=" * 60)

best_path = Path(CONFIG["project"]) / CONFIG["name"] / "weights" / "best.pt"
last_path = Path(CONFIG["project"]) / CONFIG["name"] / "weights" / "last.pt"

metrics = None  # Inicializujeme pre prípad, že best_path neexistuje

if best_path.exists():
    print(f"\n✅ Best model: {best_path}")
    print(f"   Size: {best_path.stat().st_size / 1024 / 1024:.1f} MB")
    
    # Load and evaluate
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=f"{dataset_dir}/data.yaml", device=CONFIG["device"])
    
    print(f"\n📈 Metrics:")
    print(f"  mAP50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"  mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")
    
    # Per-class metrics
    print(f"\n📊 Per-class mAP50:")
    class_names = ['Caries', 'Crown', 'Filling', 'Implant', 'Periapical-lesion']
    for i, name in enumerate(class_names):
        if i < len(metrics.box.maps):
            print(f"  {name}: {metrics.box.maps[i]:.4f}")
else:
    print("❌ No best.pt found")

# Store metrics globally for notification cell
globals()['training_metrics'] = metrics

## 💾 Export Model

In [ ]:
# Create submission directory
submission_dir = Path("/kaggle/working/submission")
submission_dir.mkdir(exist_ok=True)

# Copy best model
if best_path.exists():
    shutil.copy2(best_path, submission_dir / "best.pt")
    print(f"✅ Copied best.pt to {submission_dir}")
    
    # Copy config
    shutil.copy2(f"{dataset_dir}/data.yaml", submission_dir / "data.yaml")
    
    # Create README
    mAP50_val = metrics.box.map50 if metrics else 0.0
    mAP_val = metrics.box.map if metrics else 0.0
    readme = f"""# Dental Disease Detection Model
    
## Training Info
|- Model: {CONFIG['model']}
|- Dataset: {dataset_dir}
|- Epochs: {CONFIG['epochs']}
|- Image size: {CONFIG['imgsz']}
|- Batch size: {CONFIG['batch']}
    
## Results
|- mAP50: {mAP50_val:.4f}
|- mAP50-95: {mAP_val:.4f}
    
## Classes
0: Caries
1: Crown
2: Filling
3: Implant
4: Periapical-lesion
    
## Usage
```python
from ultralytics import YOLO
model = YOLO('best.pt')
results = model.predict('xray.jpg')
```
"""
    (submission_dir / "README.md").write_text(readme)
    
    # Create submission zip
    !cd /kaggle/working && zip -r submission.zip submission/
    print(f"\n📦 Submission ready: /kaggle/working/submission.zip")
else:
    print("❌ No model to export")

## 🔔 Notification
Pošle notifikáciu na ntfy.sh keď tréning skončí

In [ ]:
# Send completion notification via ntfy.sh
import requests

NTFY_TOPIC = "dental-ai-training-1784621752"
WEBHOOK_URL = f"https://ntfy.sh/{NTFY_TOPIC}"

try:
    if best_path.exists() and metrics:
        model_size = best_path.stat().st_size / 1e6
        msg = f"✅ Dental YOLOv8x training COMPLETE!\nmAP50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)\nmAP50-95: {metrics.box.map:.4f}\nModel: {model_size:.1f} MB"
    elif best_path.exists():
        model_size = best_path.stat().st_size / 1e6
        msg = f"✅ Dental YOLOv8x training COMPLETE!\nModel: {model_size:.1f} MB\n(No metrics - check notebook output)"
    else:
        msg = "❌ Dental YOLOv8x training FAILED - no best.pt found"
    
    requests.post(WEBHOOK_URL, 
        data=msg.encode('utf-8'),
        headers={"Title": "Dental AI Training", "Priority": "high", "Tags": "tooth,rocket"},
        timeout=10
    )
    print(f"📬 Notification sent to ntfy.sh/{NTFY_TOPIC}")
except Exception as e:
    print(f"⚠️ Notification failed: {e}")

# Also print the URL for manual checking
print(f"\n🔔 Check results: https://ntfy.sh/{NTFY_TOPIC}")
print(f"📥 Download model: kaggle kernels output eriksmite/dental-yolov8-training -p ./results")